# Tutorial 20: Magnetic-state dataset for physical hysteresis retrieval

Generate **24 states at the Co L3 absorption maximum**: two saturated up, two saturated down, ten distinct skyrmion patterns, and ten distinct labyrinth patterns. Each state has CR and CL observations. The experimental parameters are copied from Tutorial 12; only the magnetic texture and acquisition noise realization vary between states. Saturation replicates have identical magnetic truth and independent detector noise.

One numbered HDF5 file per state matches `simulation_hysteresis/reconstruction_inputs.py` in FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS. Files 0 and 2 are the +1 and -1 calibration anchors. Outputs include ideal, detected, and detected-without-beamstop holograms, centered object-plane support, beamstop opacity, and a binary exclusion mask (`True` means excluded).

Run all cells to generate a new run under `outputs/20_magnetic_state_dataset/`. The full 2048×2048 sample mesh and 50 detector frames from Tutorial 12 are retained; files are written one state at a time. This is a synthetic collection of textures, not a field-driven hysteresis trajectory.


In [ ]:
import gc
import sys
from pathlib import Path
from types import SimpleNamespace

# Make the repository's src-layout packages importable without installing them.
repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "scattering_calculator").is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError("Could not find src/scattering_calculator above the notebook working directory")
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import numpy as np
import h5py
import matplotlib.pyplot as plt

get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams["figure.constrained_layout.use"] = True

from scattering_calculator.sample_generator import pattern_generator
from scattering_calculator.simulation_pipelines import simulation_configuration as sim
import json
import hashlib
from datetime import datetime, timezone
from uuid import uuid4


from scattering_calculator.simulation_pipelines import ExperimentConfig, ScatteringExperiment, SimulationConfig
experiment = ExperimentConfig()


## Fixed experimental settings

The absorption energy changes the optical constants; the 780 eV mesh and detector projection reference stay as in Tutorial 12. Labyrinth parameters stay the same except for their random seed. Skyrmions use a 270 nm mean diameter and the same 4 nm wall width.


In [ ]:
pols=["CR","CL"]
# Use the maximum tabulated charge absorption beta on the Co L3 edge.
co_beta_path = repo_root / "src/scattering_calculator/database/material_parameter/refractive_indexes/Co/Co_beta.txt"
co_beta = np.loadtxt(co_beta_path, skiprows=2)
l3 = co_beta[(co_beta[:, 0] >= 775) & (co_beta[:, 0] <= 785)]
absorption_energy_eV = float(l3[np.argmax(l3[:, 1]), 0])
energy_values_eV = np.array([absorption_energy_eV])
RUN_DATASET = True
PLOT_LAYER_BY_LAYER = False  # Optional large diagnostic fields; physics is unchanged.
OUTPUT_ROOT = repo_root / "outputs" / "20_magnetic_state_dataset"
BEAMSTOP_THRESHOLD = 0.0
# Exact same mesh and detector projection reference as Tutorial 12.
detector_propagation_method = "fraunhofer"
# Reference energy always sets the fixed sample mesh.
reference_energy_eV = 780
# True: project every energy at reference_energy_eV for a common image scale.
# False: detector projection follows each swept energy (physical scaling).
FIX_DETECTOR_PROJECTION_ENERGY = True

# Use Jones for the full polarization interaction. Scalar is faster and can be
# used when the selected polarization is an eigenmode of the local interaction.
propagator_method = "Scalar"
scalar_refractive_index_lazy = True

# Jones-only is fast and already includes the rank-zero longitudinal phase.
# Set propagate=True to include multislice free-space propagation between layers.
propagate = True
# Optional two-stage experiment: freeze the complex field just after SiN.
# The mask stage ALWAYS propagates; propagate above then controls only the film.
FIX_MASK_ILLUMINATION = False
mask_illumination_energy_eV = 780.0
jones_apply_zero_order_phase = True
multislice_propagation_roi = True
multislice_propagation_roi_padding_px = 15
multislice_propagation_roi_merge_overlaps = False
propagation_padding_px = 10
propagation_padding_mode = "edge"
propagation_absorber_width_px = 5
propagation_absorber_strength = 6.0
propagation_absorber_profile = "cosine"

# Detector and beam settings. The sample mesh is fixed from reference_energy_eV.
detector_shape = (1024,1024)
detector_center = tuple(np.array(detector_shape) // 2)
# oversampling=2 creates a 2048x2048 sample grid; oversampling=4 creates a
# 4096x4096 sample grid and can exceed RAM during a single energy point.
oversampling = 2
ignore_flat_detector_curvature = False
photon_flux = 5e8
coherence_length = (25000e-6, 25000e-6)

# Fixed sample, aperture, illumination, and magnetic texture.
recipe = "[Au(200)/Cr(5)]x20/SiN(100)/Co(20)Pt(20)"
#recipe = "[Au(200)/Cr(5)]x20/SiN(0.5)/Pt(20)"
recipe = "[Au(90)/Cr(10)]x20/SiN(100)/[Co(8)Pt(8)]x2"
#recipe = "[Au(45)/Cr(5)]x60/SiN(0.5)/Co(20)Pt(20)"


sample_name = "cobalt_l_edge_energy_sweep_tutorial"
illumination_center = (0.0, 0.0)
illumination_focus_distance = 1e-3
illumination_fwhm = 50.e-6
illumination_alpha_beam = (0.0, 0.0)

aperture_types = ["OH", "RH", "RH"]
aperture_radii = [500e-9,  30e-9, 10e-9]
aperture_centers = [(0.0, 0.0), (-1200e-9, -1200e-9), (1200e-9, -1200e-9)]
aperture_sigmas = [4e-9, 2e-9, 2e-9]
aperture_angles = [0.0, 0.0,0]
aperture_ellipticities = [1.0, 1, 1.0]
aperture_roughnesses = [0.05,0.01, 0.01]#[0.02,0.02,0.02]
aperture_roughness_modes = [(3, 10),(3, 10), (3, 10)]#[(0, 0),(3, 10), (3, 10)]
aperture_seeds = [1,2, 4]
aperture_top_radius_factors = [1.2 , 5., 5.]

# The object image in an FTH reconstruction is not centered at the origin.
# It is displaced by the reference-hole/object-hole separation. Use the first
# RH by default; flip the sign if you want the conjugate/twin object image.
fth_reference_aperture_index = 1
fth_reconstruction_shift_sign = 1

# Domain averages ignore pixels close to a domain wall, where mz is neither
# clearly positive nor clearly negative after smoothing.
xmcd_domain_threshold = 0.9

pattern_type = "binary_labyrinth_pattern"
pattern_config = {
    'period': 1.0e-07,
    "sigma": 4e-9,
    "domain_conversion": "soft",
    "softness": 1.2,
    "n_steps": 60,
    "seed": 4,
    "use_gpu": False,
}


In [ ]:
def make_xray_config(energy_eV, pol=pols[0]):
    experiment = ExperimentConfig()
    # STANDARD SETUP: 1. xray
    experiment.xray = xray_config = sim.XRayConfig(
        energy=float(energy_eV),
        photon_flux=photon_flux,
        pol=pol,
        coherence_length=coherence_length,
    )
    xray_config.setup()
    return xray_config


def make_detector_config(noise_seed=20):
    experiment = ExperimentConfig()
    # STANDARD SETUP: 9. beamstop
    experiment.beamstop = beamstop_config = sim.BeamstopConfig(
        bs_method="circular",
        bs_detector_distance=0.010,
        bs_center=detector_center,
        bs_config={
            "radius": 90e-6,
            "sigma": 1e-6,
            "wire_width": 5e-6,
            "wire_bend": 30e-6,
            "angle": np.deg2rad(25),
            "antialias": 2,
            "seed": 4,
        },
    )
    # STANDARD SETUP: 8. detector
    experiment.detector = detector_config = sim.DetectorConfig(
        detector_propagation_method=detector_propagation_method,
        shape=detector_shape,
        pixel_size=20e-6,
        sample_to_detector_distance=0.10,
        detector_center=detector_center,
        detector_params={
            "readout_noise_average": 10,
            "readout_noise_sigma": 1,
            "detector_threshold": 64e3,
            "counts_per_photon": 200,
            "quantum_efficiency": 1.0,
            "noise_seed": int(noise_seed),
        },
        measurement_config={
            "number_frames": 50,
            "max_counts_per_image": 64e3,
            "exposure_time": 1.0,
        },
        beamstop_config=beamstop_config,
        use_detector_pixel_footprint=True,
        detector_pixel_footprint_samples=oversampling,
        ignore_flat_detector_curvature=(ignore_flat_detector_curvature if detector_propagation_method == "fraunhofer" else False),
    )
    detector_config.setup()
    return detector_config


reference_xray_config = make_xray_config(reference_energy_eV)
reference_detector_config = make_detector_config()
real_space_pixel_size = reference_detector_config.calc_realspace_resolution(reference_xray_config.beam_params) / oversampling
sample_shape = np.array([0, oversampling * detector_shape[0], oversampling * detector_shape[1]], dtype=int)

magnetic_pattern_2d = np.ones(tuple(sample_shape[1:]), dtype=np.float32)
print(f"Co L3 absorption energy: {absorption_energy_eV:.8f} eV")
print(f"Sample grid: {tuple(sample_shape[1:])}; pixel size: {real_space_pixel_size * 1e9:.3f} nm")


## Generate reproducible magnetic states

Uniqueness is checked inside the object hole, where the magnetic texture is observed. Seeds differ across the ten states of each morphology. Patterns are regenerated on demand to avoid retaining 24 full sample arrays.


In [ ]:
states = [
    dict(name="saturated_up_01", kind="saturated", mz=1.0),
    dict(name="saturated_up_02", kind="saturated", mz=1.0),
    dict(name="saturated_down_01", kind="saturated", mz=-1.0),
    dict(name="saturated_down_02", kind="saturated", mz=-1.0),
] + [dict(name=f"skyrmion_{i+1:02d}", kind="skyrmion", seed=100+i) for i in range(10)]   + [dict(name=f"labyrinth_{i+1:02d}", kind="labyrinth", seed=200+i) for i in range(10)]

y, x = np.indices(tuple(sample_shape[1:]), dtype=float)
oh = aperture_types.index("OH")
object_mask = (((y-sample_shape[1]/2)*real_space_pixel_size-aperture_centers[oh][0])**2
             + ((x-sample_shape[2]/2)*real_space_pixel_size-aperture_centers[oh][1])**2
             <= aperture_radii[oh]**2)
del y, x

def state_pattern(state):
    if state["kind"] == "saturated":
        return np.full(tuple(sample_shape[1:]), state["mz"], dtype=np.float32)
    if state["kind"] == "labyrinth":
        method = "binary_labyrinth_pattern"
        parameters = {**pattern_config, "seed": state["seed"]}
    else:
        method = "disordered_skyrmion_lattice_pattern"
        parameters = dict(stripe_width=100e-9, sigma=4e-9,
                          skyrmion_density=0.45, diameter_spread=40e-9,
                          ellipticity=(0.75, 1.35), roughness=0.03,
                          placement_center=tuple(sample_shape[1:]/2),
                          placement_radius=aperture_radii[oh]+270e-9,
                          seed=state["seed"])
    cfg = sim.MagneticPatternConfig(pattern_type_method=method,
        shape=tuple(sample_shape[1:]), real_space_pixel_size=real_space_pixel_size,
        pattern_config=parameters)
    cfg.create_pattern()
    return np.asarray(cfg.magnetic_pattern, dtype=np.float32)

fig, axes = plt.subplots(4, 6, figsize=(16, 10), layout="constrained")
pattern_hashes = set()
ys, xs = np.where(object_mask)
preview_roi = (slice(ys.min(), ys.max()+1), slice(xs.min(), xs.max()+1))
for ax, state in zip(axes.flat, states):
    pattern = state_pattern(state)
    digest = hashlib.sha256(pattern[object_mask].tobytes()).hexdigest()
    if state["kind"] != "saturated":
        if digest in pattern_hashes:
            raise RuntimeError(f"Duplicate object-hole pattern: {state['name']}")
        if not (pattern[object_mask].min() < -0.5 and pattern[object_mask].max() > 0.5):
            raise RuntimeError(f"Missing magnetic domains: {state['name']}")
        pattern_hashes.add(digest)
    state["object_pattern_sha256"] = digest
    state["object_mean_mz"] = float(pattern[object_mask].mean())
    ax.imshow(pattern[preview_roi], cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_title(state["name"], fontsize=9); ax.axis("off")
    del pattern
assert len(states) == 24 and len(pattern_hashes) == 20
plt.show()


## Simulation helpers from Tutorial 12

These preserve the sample stack, tapered rough apertures, Gaussian illumination, scalar multislice propagation, detector projection, and detector effects.


In [ ]:
def frame_average(array):
    if isinstance(array, h5py.Dataset) and array.ndim == 3:
        # Read one frame at a time, including when analysing older saved runs.
        mean = np.zeros(array.shape[1:], dtype=np.result_type(array.dtype, np.float64))
        for index in range(array.shape[0]):
            mean += array[index]
        return mean / array.shape[0]
    array = np.asarray(array)
    return array.mean(axis=0) if array.ndim == 3 else array


def fth_reconstruct(hologram):
    """Reconstruct over the spatial axes without shifting a frame axis."""
    axes = (-2, -1)
    return np.fft.fftshift(
        np.fft.fft2(np.fft.fftshift(hologram, axes=axes), axes=axes),
        axes=axes,
    )


def symmetric_limits(data, percentile=99.0):
    vmax = np.nanpercentile(np.abs(data), percentile)
    return -float(vmax or 1.0), float(vmax or 1.0)


def continuous_phase(values):
    """Return a phase curve without artificial 2*pi branch-cut jumps."""
    phase = np.unwrap(np.angle(values))
    offset = 2 * np.pi * np.round(np.nanmedian(phase) / (2 * np.pi))
    return phase - offset


def continuous_complex_log_ratio(numerator, denominator, eps=1e-30):
    """Compute log(numerator / denominator) with an unwrapped imaginary part."""
    ratio = (np.asarray(numerator) + eps) / (np.asarray(denominator) + eps)
    return np.log(np.abs(ratio)) + 1j * continuous_phase(ratio)


def resample_nearest_to_shape(array, shape):
    """Nearest-neighbor resample a 2D mask/image to a requested shape."""
    array = np.asarray(array)
    if array.shape == tuple(shape):
        return array
    y_idx = np.rint(np.linspace(0, array.shape[0] - 1, shape[0])).astype(int)
    x_idx = np.rint(np.linspace(0, array.shape[1] - 1, shape[1])).astype(int)
    return array[np.ix_(y_idx, x_idx)]


def object_hole_crop_slices(shape, pixel_size, radius_factor=1.0):
    oh_index = aperture_types.index("OH")
    center_y_m, center_x_m = aperture_centers[oh_index]
    return crop_slices_around_position(shape, pixel_size, (center_y_m, center_x_m), aperture_radii[oh_index] * radius_factor)


def fth_object_image_crop_slices(shape, pixel_size, radius_factor=1.0):
    """Crop the FTH object image displaced by the selected RH-OH vector.

    In the exit wave the object hole is centered at its real-space OH position.
    In the FTH reconstruction, the object image appears in the cross-correlation
    term, shifted from the reconstruction origin by the reference-hole/object-hole
    separation. The sign selects which twin image to inspect.
    """
    oh_index = aperture_types.index("OH")
    oh_center = np.asarray(aperture_centers[oh_index], dtype=float)
    rh_center = np.asarray(aperture_centers[fth_reference_aperture_index], dtype=float)
    reconstruction_center = fth_reconstruction_shift_sign * (rh_center - oh_center)
    return crop_slices_around_position(
        shape,
        pixel_size,
        reconstruction_center,
        aperture_radii[oh_index] * radius_factor,
    )


def crop_slices_around_position(shape, pixel_size, center_m, radius_m):
    center_y_m, center_x_m = center_m
    cy = int(np.clip(round(shape[-2] / 2 + center_y_m / pixel_size), 0, shape[-2] - 1))
    cx = int(np.clip(round(shape[-1] / 2 + center_x_m / pixel_size), 0, shape[-1] - 1))
    half_width = max(4, int(np.ceil(radius_m / pixel_size)))
    y0 = max(0, cy - half_width)
    y1 = min(shape[-2], cy + half_width + 1)
    x0 = max(0, cx - half_width)
    x1 = min(shape[-1], cx + half_width + 1)
    return slice(y0, y1), slice(x0, x1)


sample_oh_slices = object_hole_crop_slices(sample_shape[1:], real_space_pixel_size, radius_factor=1.0)


def build_sample_and_aperture(xray_config):
    experiment = ExperimentConfig()
    # STANDARD SETUP: 3. sample
    # STANDARD SETUP: 2. simulation
    experiment.simulation = SimulationConfig(shape=tuple((sample_shape.copy())[-2:]), real_space_pixel_size=real_space_pixel_size)
    experiment.sample = sample_config = sim.SampleConfig(
        recipe=recipe,
        sample_shape=sample_shape.copy(),
        real_space_pixel_size=real_space_pixel_size,
        xray_config=xray_config,
        sample_name=sample_name,
    )
    sample_config.setup()

    magnetization = pattern_generator.map_magnetization_to_3d(
        magnetic_pattern_x=np.zeros_like(magnetic_pattern_2d),
        magnetic_pattern_y=np.sqrt(np.clip(1 - np.abs(magnetic_pattern_2d) ** 2, 0, 1)),
        magnetic_pattern_z=magnetic_pattern_2d,
        nr_repeats=sample_config.sample_structure.sample_shape[0],
    )
    sample_config.assign_magnetic_pattern(magnetization)

    layer_thicknesses = sample_config.sample_structure.layer_thicknesses
    membrane_index = sample_config.sample_structure.layer_names.index("SiN")
    aperture_taper_depth = float(np.sum(layer_thicknesses[: max(0, membrane_index - 2)]))
    thickness_oh = float(np.sum(layer_thicknesses[:membrane_index]))

    # STANDARD SETUP: 5. aperture
    experiment.aperture = aperture_config = sim.FrontApertureConfig(
        aperture_method="FTH_circular",
        aperture_shape=sample_config.sample_structure.sample_shape,
        real_space_pixel_size=real_space_pixel_size,
        aperture_thicknesses=layer_thicknesses,
        aperture_layer_names=sample_config.sample_structure.layer_names,
        use_roi=True,
        aperture_config={
            "apertures_type": aperture_types,
            "apertures_radius": aperture_radii,
            "apertures_center": aperture_centers,
            "apertures_sigma": aperture_sigmas,
            "apertures_angle": aperture_angles,
            "apertures_ellipticity": aperture_ellipticities,
            "apertures_roughness": aperture_roughnesses,
            "apertures_roughness_modes": aperture_roughness_modes,
            "apertures_seed": aperture_seeds,
            "apertures_top_radius_factor": aperture_top_radius_factors,
            "aperture_taper_depth": aperture_taper_depth,
            "thickness_OH": thickness_oh,
        },
    )
    aperture_config.setup()
    aperture_mask = aperture_config.return_aperture()
    sample_config.assign_aperture_mask(aperture_mask)

    if propagator_method == "Scalar":
        sample_config.sample_structure.calculate_final_scalar_refractive_index(
            pol=pols[0],
            use_aperture_roi=True,
            compact=True,
            lazy=scalar_refractive_index_lazy,
        )
    else:
        contrast_beam_direction = sim.light_beam.beam_direction_from_alpha(illumination_alpha_beam)
        if np.allclose(contrast_beam_direction, [0.0, 0.0, 1.0], atol=1e-14, rtol=0.0):
            contrast_beam_direction = None
        sample_config.sample_structure.calculate_final_dielectric_tensor(
            use_aperture_roi=True,
            compact=True if contrast_beam_direction is None else False,
            beam_direction=contrast_beam_direction,
        )
    return sample_config, aperture_config, aperture_mask


def membrane_split_index(structure):
    """Split the original, untilted layer stack immediately after its SiN layer."""
    if structure.layer_names.count("SiN") != 1:
        raise ValueError("Fixed mask illumination requires exactly one SiN layer.")
    split = structure.layer_names.index("SiN") + 1
    if split == len(structure.layer_names):
        raise ValueError("The recipe must contain film layers after SiN.")
    if len(structure.propagation_layer_thicknesses) != len(structure.layer_names):
        raise ValueError("Fixed mask illumination requires an untilted layer stack.")
    return split


def layer_stack_view(stack, start, stop):
    """Slice optical layers without materializing a dense mask volume."""
    if hasattr(stack, "base_index"):
        view = SimpleNamespace(
            shape=(stop - start, *stack.shape[1:]),
            base_index=stack.base_index[start:stop],
        )
        if hasattr(stack, "layer_patches"):
            view.layer_patches = lambda iz: stack.layer_patches(start + iz)
        else:
            view.patches = stack.patches[start:stop]
        return view
    if hasattr(stack, "base_diagonal"):
        return SimpleNamespace(
            shape=(stop - start, *stack.shape[1:]),
            base_diagonal=stack.base_diagonal[start:stop],
            ndim=5,
            patches=stack.patches[start:stop],
        )
    return stack[start:stop]


def stage_sample_config(sample_config, mask_stage=False):
    """Reuse the full-stack aperture geometry, selecting only this stage's optics."""
    structure = sample_config.sample_structure
    split = membrane_split_index(structure)
    start, stop = (0, split + 1) if mask_stage else (split, len(structure.layer_names))
    thicknesses = list(structure.propagation_layer_thicknesses[start:stop])
    if mask_stage:
        # The engine diffracts after every non-final slice. A zero-thickness
        # marker at the film entrance retains diffraction across the last SiN
        # slice without applying any film transmission or adding a vacuum gap.
        thicknesses[-1] = 0.0
    stage = SimpleNamespace(
        propagation_layer_thicknesses=thicknesses,
        aperture_support_regions=getattr(structure, "aperture_support_regions", None),
    )
    if propagator_method == "Scalar":
        def calculate_scalar(pol, **kwargs):
            structure.calculate_final_scalar_refractive_index(pol, **kwargs)
            stage.final_scalar_refractive_index = layer_stack_view(
                structure.final_scalar_refractive_index, start, stop
            )
        stage.calculate_final_scalar_refractive_index = calculate_scalar
    else:
        stage.final_dielectric_tensor = layer_stack_view(
            structure.final_dielectric_tensor, start, stop
        )
    return SimpleNamespace(sample_structure=stage, real_space_pixel_size=real_space_pixel_size)


def make_illumination(xray_config):
    experiment = ExperimentConfig()
    # STANDARD SETUP: 6. illumination
    experiment.illumination = illumination_config = sim.IlluminationConfig(
        XRayConfig=xray_config,
        shape=tuple(sample_shape[1:]),
        real_space_pixel_size=real_space_pixel_size,
        illumination_function="gaussian",
        illumination_config={
            "center": np.array(illumination_center),
            "distance": illumination_focus_distance,
            "fwhm": illumination_fwhm,
            "alpha_beam": illumination_alpha_beam,
        },
    )
    illumination_config.setup()
    return illumination_config


def propagation_options(do_propagate):
    return {
        "propagate": do_propagate,
        "jones_apply_zero_order_phase": jones_apply_zero_order_phase,
        "scalar_apply_zero_order_phase": jones_apply_zero_order_phase,
        "scalar_refractive_index_lazy": scalar_refractive_index_lazy,
        "propagation_padding_px": propagation_padding_px,
        "propagation_padding_mode": propagation_padding_mode,
        "propagation_absorber_width_px": propagation_absorber_width_px,
        "propagation_absorber_strength": propagation_absorber_strength,
        "propagation_absorber_profile": propagation_absorber_profile,
        "multislice_propagation_roi": multislice_propagation_roi,
        "multislice_propagation_roi_padding_px": multislice_propagation_roi_padding_px,
        "multislice_propagation_roi_merge_overlaps": multislice_propagation_roi_merge_overlaps,
    }


def layer_field_writer(group, structure, pol, offset=0, stop=None):
    """Stream the actual full post-material fields; no layer stack is held in RAM."""
    if group is None:
        return None
    if "layer_names" not in group.attrs:
        group.attrs["layer_names"] = np.asarray(structure.layer_names, dtype=h5py.string_dtype())
        group.attrs["layer_thicknesses_m"] = structure.layer_thicknesses
        group.attrs["pixel_size_m"] = float(real_space_pixel_size)
        group.attrs["recipe"] = recipe
        group.attrs["plane"] = "after material interaction, before propagation to next layer"
        group.attrs["propagator_method"] = propagator_method
        oh = aperture_types.index("OH")
        group.attrs["oh_center_m"] = aperture_centers[oh]
        group.attrs["oh_radius_m"] = aperture_radii[oh]
        group.attrs["oh_top_radius_factor"] = aperture_top_radius_factors[oh]
    fields = group.require_group(pol)
    def write_layer(iz, field):
        index = offset + iz
        if stop is not None and index >= stop:
            return  # Exclude the zero-thickness marker in the fixed-mask stage.
        fields.create_dataset(f"{index:05d}", data=field, compression="lzf", shuffle=True)
    return write_layer


def prepare_fixed_mask_illumination(layer_group=None):
    """Calculate once per sweep, retaining a complex field for each polarization."""
    experiment = ExperimentConfig()
    if propagator_method not in ("Scalar", "Jones"):
        raise ValueError("Fixed mask illumination supports Scalar and Jones propagation.")
    xray = make_xray_config(mask_illumination_energy_eV)
    sample, _, _ = build_sample_and_aperture(xray)
    stage = stage_sample_config(sample, mask_stage=True)
    illumination = make_illumination(xray)
    fields = {}
    for pol in pols:
        illumination.update_polarization(pol)
        # STANDARD SETUP: 7. propagation
        experiment.propagation = propagator = sim.SamplePropagatorConfig(
            SampleConfig=stage, IlluminationConfig=illumination,
            propagator_method=propagator_method,
            propagator_config={**propagation_options(True), "calculate_farfield": False,
                               "layer_callback": layer_field_writer(
                                   layer_group, sample.sample_structure, pol,
                                   stop=membrane_split_index(sample.sample_structure))},
        )
        propagator.setup()
        # Keep Jones components where available; never reduce them to an
        # amplitude/phase surrogate or normalize away mask absorption.
        fields[pol] = np.array(propagator.wavefront.exit_wave, copy=True)
        del propagator
    return fields


def simulate_one_energy(energy_eV, noise_seed_base=1000, fixed_mask_fields=None, layer_group=None):
    experiment = ExperimentConfig()
    xray_config = make_xray_config(energy_eV)
    detector_config = make_detector_config(noise_seed=noise_seed_base)
    projection_beam_params = (
        reference_xray_config.beam_params
        if FIX_DETECTOR_PROJECTION_ENERGY else xray_config.beam_params
    )
    detector_config.calc_realspace_resolution(projection_beam_params)
    sample_config, aperture_config, aperture_mask = build_sample_and_aperture(xray_config)

    illumination_config = make_illumination(xray_config)
    if FIX_MASK_ILLUMINATION and fixed_mask_fields is None:
        raise ValueError("Prepare fixed mask illumination before running the sweep.")
    propagation_sample = (
        stage_sample_config(sample_config) if FIX_MASK_ILLUMINATION else sample_config
    )

    hologram_config = sim.HologramConfig(
        sample_x=sample_config.sample_structure.x,
        sample_y=sample_config.sample_structure.y,
        detector_layout=detector_config.detector_layout,
    )


    for pol_index, pol in enumerate(pols):
        detector_config.detector_params["noise_seed"] = int(noise_seed_base) + pol_index
        illumination_config.update_polarization(pol)
        if FIX_MASK_ILLUMINATION:
            field = fixed_mask_fields[pol]
            illumination_config.illumination.illumination_jones = (
                sim.light_beam.scalar_to_jones(field, pol)
                if field.ndim == 2 else field.copy()
            )
        # STANDARD SETUP: 7. propagation
        experiment.propagation = propagator_config = sim.SamplePropagatorConfig(
            SampleConfig=propagation_sample,
            IlluminationConfig=illumination_config,
            propagator_method=propagator_method,
            propagator_config={**propagation_options(propagate),
                               "layer_callback": layer_field_writer(
                                   layer_group, sample_config.sample_structure, pol,
                                   offset=(membrane_split_index(sample_config.sample_structure)
                                           if FIX_MASK_ILLUMINATION else 0))},
        )
        propagator_config.setup()
        detector_config.assign_propagated_wavefront(propagator_config)
        # Use the selected energy for detector projection only.
        detector_config.detect_hologram(
            projection_beam_params=projection_beam_params,
        )

        hologram_config.add_exit_waves({pol: propagator_config.return_scalar_wavefield()})
        hologram_config.add_holograms({pol: detector_config.return_ideal_hologram()}, source="ideal")
        hologram_config.add_holograms({pol: detector_config.return_detected_hologram(store_no_beamstop=True)}, source="detected")
        hologram_config.add_holograms({pol: detector_config.return_detected_hologram_without_beamstop()}, source="detected_no_beamstop")
        del propagator_config

    hologram_config.compute_differences(positive=pols[0], negative=pols[1])
    hologram_config.compute_sums(left=pols[0], right=pols[1])
    # Keep native images; their physical pixel size follows the projection energy.
    fth_hologram_scale = 1.0
    # Reconstruct frame by frame during HDF5 writing, after releasing the sample.
    eps = 1e-30
    xmcd_ratio = (hologram_config.exit_waves[pols[0]] + eps) / (hologram_config.exit_waves[pols[1]] + eps)
    xmcd_log = np.log(xmcd_ratio)
    native_reconstruction_pixel_size = detector_config.detector_layout.real_space_resolution
    reconstruction_pixel_size = native_reconstruction_pixel_size * fth_hologram_scale
    supportmask = aperture_config.create_supportmask(
        output_shape=detector_config.detector_layout.detector_shape,
        output_pixel_size=reconstruction_pixel_size,
    )
    recon_oh_slices = fth_object_image_crop_slices(detector_shape, reconstruction_pixel_size, radius_factor=1.0)
    oh_index = aperture_types.index("OH")
    fth_shift_m = fth_reconstruction_shift_sign * (
        np.asarray(aperture_centers[fth_reference_aperture_index], dtype=float)
        - np.asarray(aperture_centers[oh_index], dtype=float)
    )

    total_ideal_intensity = float(np.sum(hologram_config.ideal_holograms["sum"]))
    total_detected_counts = float(np.sum(hologram_config.detected_holograms["sum"]))

    # These objects own large sample meshes and propagation work arrays. None of
    # the analysis cells below uses them, so do not retain them across energies.
    hologram_config.sample_x = None
    hologram_config.sample_y = None
    hologram_config.detector_layout = None

    return {
        "energy_eV": float(energy_eV),
        "supportmask": supportmask,
        "beamstop_mask": np.asarray(detector_config.detector_layout.beamstop).copy(),
        "holograms": hologram_config,
        "xmcd_ratio": xmcd_ratio,
        "xmcd_log": xmcd_log,
        "sample_oh_slices": sample_oh_slices,
        "recon_oh_slices": recon_oh_slices,
        "fth_hologram_scale": fth_hologram_scale,
        "reconstruction_pixel_size": reconstruction_pixel_size,
        "fth_reconstruction_shift_m": fth_shift_m,
        "total_ideal_intensity": total_ideal_intensity,
        "total_detected_counts": total_detected_counts,
    }


## Write one HDF5 file per state

Each file is published only after it is complete. The manifest records state order and acquisition seeds. The soft beamstop opacity is saved separately from `mask_pixel`; the reconstruction loader derives its exclusion mask from that opacity and its chosen threshold. No intensity normalization is applied.


In [ ]:
def write_array(group, name, value):
    group.create_dataset(name, data=np.asarray(value), compression="lzf", shuffle=True)

# Save all controls needed to reproduce the fixed physical experiment.
control_names = (
    "recipe", "reference_energy_eV", "absorption_energy_eV", "pols", "oversampling",
    "photon_flux", "coherence_length", "propagator_method", "propagate",
    "FIX_MASK_ILLUMINATION", "mask_illumination_energy_eV", "FIX_DETECTOR_PROJECTION_ENERGY",
    "illumination_center", "illumination_focus_distance", "illumination_fwhm", "illumination_alpha_beam",
    "aperture_types", "aperture_radii", "aperture_centers", "aperture_sigmas", "aperture_angles",
    "aperture_ellipticities", "aperture_roughnesses", "aperture_roughness_modes", "aperture_seeds",
    "aperture_top_radius_factors", "ignore_flat_detector_curvature", "pattern_config",
)
configuration = {name: globals()[name] for name in control_names}
configuration["propagation"] = propagation_options(propagate)
configuration["detector"] = dict(shape=detector_shape, pixel_size=20e-6,
    sample_to_detector_distance=0.10, detector_center=detector_center,
    detector_propagation_method=detector_propagation_method,
    detector_params=reference_detector_config.detector_params,
    measurement_config=reference_detector_config.measurement_config,
    use_detector_pixel_footprint=True, detector_pixel_footprint_samples=oversampling)
configuration["beamstop"] = dict(bs_method="circular", bs_detector_distance=0.010,
    bs_center=detector_center, bs_config=reference_detector_config.beamstop_config.bs_config)
configuration["skyrmion"] = dict(stripe_width=270e-9, sigma=4e-9, skyrmion_density=0.25,
    diameter_spread=40e-9, ellipticity=(0.75, 1.35), roughness=0.05,
    placement_center=tuple(sample_shape[1:]/2), placement_radius=aperture_radii[oh]+270e-9)

def json_default(value):
    return np.asarray(value).tolist()

if RUN_DATASET:
    run_id = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ_") + uuid4().hex[:8]
    RUN_DIRECTORY = OUTPUT_ROOT / run_id
    RUN_DIRECTORY.mkdir(parents=True, exist_ok=False)
    fixed_mask_fields = prepare_fixed_mask_illumination() if FIX_MASK_ILLUMINATION else None
    shared_support = None
    records = []
    for index, state in enumerate(states):
        print(f"[{index+1:02d}/24] {state['name']}", flush=True)
        magnetic_pattern_2d = state_pattern(state)
        noise_seed = 1000 + 10*index
        destination = RUN_DIRECTORY / f"{index:02d}_{state['name']}.h5"
        partial = destination.with_suffix(".partial")
        with h5py.File(partial, "x") as f:
            layers = f.create_group("layer_fields") if PLOT_LAYER_BY_LAYER else None
            result = simulate_one_energy(absorption_energy_eV, noise_seed_base=noise_seed,
                                         fixed_mask_fields=fixed_mask_fields, layer_group=layers)
            support = np.asarray(result["supportmask"], dtype=bool)
            opacity = np.asarray(result["beamstop_mask"])
            assert support.shape == opacity.shape == detector_shape
            if shared_support is None: shared_support = support.copy()
            elif not np.array_equal(shared_support, support):
                raise RuntimeError("Support changed between magnetic states")
            f.attrs["configuration_json"] = json.dumps(configuration, default=json_default)
            f.attrs["energy_eV"] = absorption_energy_eV
            f.attrs["state_name"] = state["name"]
            f.attrs["state_kind"] = state["kind"]
            f.attrs["state_json"] = json.dumps(state)
            f.attrs["object_mean_mz"] = state["object_mean_mz"]
            f.attrs["noise_seed_base"] = noise_seed
            f.attrs["object_pattern_sha256"] = state["object_pattern_sha256"]
            f.attrs["reconstruction_pixel_size_m"] = result["reconstruction_pixel_size"]
            f.attrs["mask_pixel_convention"] = "True means excluded; beamstop opacity > threshold"
            write_array(f, "support_mask", support)
            masks = f.create_group("masks")
            write_array(masks, "beamstop_mask", opacity)
            write_array(f, "mask_pixel", opacity > BEAMSTOP_THRESHOLD)
            masks["mask_pixel"] = f["mask_pixel"]
            truth = f.create_group("truth")
            write_array(truth, "magnetization_z", magnetic_pattern_2d)
            h = result["holograms"]
            holograms = f.create_group("holograms")
            for name, store in (("ideal", h.ideal_holograms),
                                ("detected", h.detected_holograms),
                                ("detected_no_beamstop", h.detected_holograms_no_beamstop)):
                group = holograms.create_group(name)
                for pol in pols:
                    write_array(group, pol, store[pol])
            waves = f.create_group("exit_waves")
            for pol in pols: write_array(waves, pol, h.exit_waves[pol])
            f.attrs["complete"] = True
        partial.replace(destination)
        records.append(dict(index=index, file=destination.name, noise_seed_base=noise_seed, **state))
        (RUN_DIRECTORY / "manifest.json").write_text(json.dumps(dict(
            energy_eV=absorption_energy_eV, expected_states=24, completed_states=len(records),
            configuration=configuration, states=records), indent=2, default=json_default)+"\n")
        del result, h, magnetic_pattern_2d
        gc.collect()
    assert len(list(RUN_DIRECTORY.glob("*.h5"))) == 24
    print("Dataset saved:", RUN_DIRECTORY)
else:
    print("Generation disabled. Set RUN_DATASET=True to create the dataset.")


## Load in the physical hysteresis notebook

Set `DATA_ROOT` in `01_physical_hysteresis.ipynb` to the printed `RUN_DIRECTORY` (the directory containing the 24 `.h5` files). Keep `FILE_INDICES=None` to load every state. Choose `HOLOGRAM_SOURCE='ideal'` or `'detected_no_beamstop'` and `HELICITY='CR'` or `'CL'`.

The existing adapter accepts ideal and detected-without-beamstop sources; it does not currently accept `'detected'`, although those frames are saved here. It derives `mask_pixel` from `masks/beamstop_mask`. The reconstruction notebook currently zeroes the loaded mask; remove that override if beamstop exclusion should be used.

The optional cell below verifies compatibility with the actual adapter without running phase retrieval.


In [ ]:
# Optional adapter check; run after generation.
import importlib.util
adapter_path = Path("/Users/riccardo/Github/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS/simulation_hysteresis/reconstruction_inputs.py")
if RUN_DATASET and adapter_path.is_file():
    spec = importlib.util.spec_from_file_location("hysteresis_inputs", adapter_path)
    inputs = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(inputs)
    for source in ("ideal", "detected_no_beamstop"):
        for pol in pols:
            data = inputs.load_dataset(RUN_DIRECTORY, source=source, helicity=pol)
            assert len(data["state_labels"]) == 24
            assert set(data["saturated"].values()) == {-1.0, 1.0}
            assert data["holograms"].shape == (24, *detector_shape)
            np.testing.assert_allclose(data["truth_mean_mz"][:4], [1, 1, -1, -1])
            print("Adapter OK:", source, pol, data["holograms"].shape)
            del data
